# Verify services

Connectivity and capability probe for Neo4j, Qdrant and Bedrock. Replaces `scripts/verify_services.py`. Prints no secrets.

## Setup

Redacted readiness report — `config.py` never prints a secret.

In [1]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")          # Neo4j emits schema hints we handle
import config
print(config.report())

readiness
  neo4j      uri=neo4j+s://<redacted>.databases.neo4j.io
             user fp=6b611cde  db fp=6b611cde
             password fp=58ebef03
             configured=True
  qdrant     url=https://<redacted>.sa-east-1-0.aws.cloud.qdrant.io
             api_key fp=f3da5a0f
             configured=True
  embedding  provider=bedrock  model=amazon.titan-embed-text-v2:0  dim=1024
             configured=True
  trust      policy=v1.0  floor=0.75  min_tier=ingested


## Neo4j

Connectivity, version, existing graph size, write permission, and the procedures the specs depend on.

In [2]:
from biomedical.graph.store import GraphClient
cfg = config.load()

with GraphClient(uri=cfg.neo4j.uri, username=cfg.neo4j.username,
                 password=cfg.neo4j.password.reveal(), database=cfg.neo4j.database) as c:
    comp = c.run("CALL dbms.components() YIELD name, versions, edition "
                 "RETURN name, versions[0] AS version, edition")[0]
    print(f"{comp['name']} {comp['version']} ({comp['edition']})")
    print("nodes:", c.run("MATCH (n) RETURN count(n) AS v")[0]["v"])
    print("rels: ", c.run("MATCH ()-[r]->() RETURN count(r) AS v")[0]["v"])
    print("constraints:", c.constraint_count())
    procs = c.run("SHOW PROCEDURES YIELD name WHERE name IN "
                  "['db.index.vector.queryNodes','db.index.fulltext.queryNodes'] "
                  "RETURN collect(name) AS p")[0]["p"]
    print("vector/fulltext procedures:", sorted(procs))

Neo4j Kernel 5.27-aura (enterprise)
nodes: 7819


rels:  10630
constraints: 11


vector/fulltext procedures: ['db.index.fulltext.queryNodes', 'db.index.vector.queryNodes']


## Qdrant

In [3]:
from qdrant_client import QdrantClient
qc = QdrantClient(url=cfg.qdrant.url, api_key=cfg.qdrant.api_key.reveal(),
                  timeout=cfg.qdrant.timeout)
cols = qc.get_collections().collections
print(f"collections: {len(cols)}")
print("ours (biomedv2__*):", [c.name for c in cols if c.name.startswith("biomedv2")] or "none")

collections: 40
ours (biomedv2__*): none


## Bedrock

Listing a model is not the same as being able to invoke it. Claude 5.x is blocked on this role by marketplace eligibility; `claude-sonnet-4-6` works and is the model the evaluation uses.

In [4]:
import boto3, json, os
rt = boto3.client("bedrock-runtime", region_name=os.environ.get("AWS_DEFAULT_REGION", "us-west-2"))

try:
    r = rt.invoke_model(modelId="amazon.titan-embed-text-v2:0",
                        body=json.dumps({"inputText": "pembrolizumab", "dimensions": 1024}))
    print("titan-embed-text-v2  OK  dim =", len(json.loads(r["body"].read())["embedding"]))
except Exception as e:
    print("titan-embed-text-v2  FAILED:", type(e).__name__)

for mid in ("us.anthropic.claude-sonnet-4-6", "us.anthropic.claude-haiku-5-5"):
    try:
        out = rt.converse(modelId=mid,
                          messages=[{"role": "user", "content": [{"text": "Reply with exactly: PASS"}]}],
                          inferenceConfig={"maxTokens": 8, "temperature": 0})
        print(f"{mid:<40} OK  -> {out['output']['message']['content'][0]['text'].strip()!r}")
    except Exception as e:
        print(f"{mid:<40} {type(e).__name__}: {str(e)[:70]}")

titan-embed-text-v2  OK  dim = 1024


us.anthropic.claude-sonnet-4-6           OK  -> 'PASS'
us.anthropic.claude-haiku-5-5            AccessDeniedException: An error occurred (AccessDeniedException) when calling the Converse op
